# 01. Batch1 데이터 구조 확인

이 노트북은 **MATLAB v7.3 구조 확인 → HDF5 reference 해제 → cell-level 필드 확인 → summary 구조 확인 → cycles 구조 확인 → Vdlin 구조 확인 → processed 데이터 저장 → EDA 전달** 순서로 데이터 수준을 파악합니다. 셀 메타데이터 CSV는 앞부분에서 먼저 저장하고, 뒤에서 cycle 데이터와 공통 전압축을 별도로 저장합니다.

In [77]:
# 데이터 구조 확인 단계: 노트북이 프로젝트 가상환경의 Python을 쓰는지 확인합니다.
import sys
print(sys.executable)

/Users/u102/workspace_public/021_Data_Analytics_Mini_Projcet/Data_Analytics_MiniProject_Skala/.venv/bin/python


In [78]:
# Batch1의 HDF5 참조를 읽고 표로 정리하는 데 필요한 도구를 불러옵니다.
from pathlib import Path

import h5py
import numpy as np
import pandas as pd

In [79]:
# Batch1의 구조와 입력을 확인합니다.
DATA_DIR = Path("../data")
batch1_path = DATA_DIR / "Batch1.mat"

In [80]:
# 파일이 준비됐는지 먼저 확인합니다.
assert batch1_path.is_file()
print(batch1_path.name, batch1_path.exists())

Batch1.mat True


## MATLAB v7.3과 Batch1 구조

`Batch1.mat`은 MATLAB v7.3 형식이므로 `scipy.io.loadmat` 대신 HDF5 리더인 `h5py`로 엽니다. 최상위에는 `batch`, `batch_date`, `#refs#`, `#subsystem#`이 있으며 분석 대상은 `batch`입니다. `batch`의 주요 필드는 `Vdlin`, `barcode`, `channel_id`, `cycle_life`, `cycles`, `policy`, `policy_readable`, `summary`입니다.

In [81]:
# MATLAB v7.3 파일은 HDF5 형식이므로 h5py로 최상위 구조를 확인합니다.
with h5py.File(batch1_path, "r") as f:
    print(list(f.keys()))

['#refs#', '#subsystem#', 'batch', 'batch_date']


In [82]:
# batch의 필드별 크기와 자료형을 살펴 셀 수와 참조 저장 방식을 파악합니다.
with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    for key in batch.keys():
        obj = batch[key]
        print(
            f"{key:15} "
            f"type={type(obj).__name__:10} "
            f"shape={obj.shape} "
            f"dtype={obj.dtype}"
        )

Vdlin           type=Dataset    shape=(46, 1) dtype=object
barcode         type=Dataset    shape=(46, 1) dtype=object
channel_id      type=Dataset    shape=(46, 1) dtype=object
cycle_life      type=Dataset    shape=(46, 1) dtype=object
cycles          type=Dataset    shape=(46, 1) dtype=object
policy          type=Dataset    shape=(46, 1) dtype=object
policy_readable type=Dataset    shape=(46, 1) dtype=object
summary         type=Dataset    shape=(46, 1) dtype=object


## HDF5 object reference 해제

MATLAB v7.3 데이터는 실제 값 대신 HDF5 object reference를 담는 경우가 많습니다. 아래에서 한 값을 직접 역참조한 뒤, `read_h5_value()`로 숫자 하나, 문자열, 배열을 읽어 cell-level 메타데이터 추출에 사용합니다.

In [83]:
# cycle_life 한 건을 역참조해 실제 값이 참조 뒤에 저장됐음을 확인합니다.
with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    ref = batch["cycle_life"][0, 0]
    print(ref)
    print(type(ref))

    obj = f[ref]
    print(type(obj))
    print(obj.shape)
    print(obj[()])

<HDF5 object reference>
<class 'h5py.h5r.Reference'>
<class 'h5py._hl.dataset.Dataset'>
(1, 1)
[[1190.]]


In [84]:
# HDF5 reference를 실제 dataset으로 풀어 cell-level 값 추출에 재사용합니다.
# 여러 셀에 반복 사용할 참조 해제 함수입니다. 숫자, MATLAB 문자, 배열을 구분합니다.
def read_h5_value(f, ref):
    """
    HDF5 object reference를 실제 값으로 변환
    """
    obj = f[ref]
    value = obj[()]

    # 숫자 1개짜리 배열
    if value.size == 1:
        return np.squeeze(value).item()

    # MATLAB 문자열은 uint16 배열로 저장되는 경우가 많음
    if np.issubdtype(value.dtype, np.integer):
        try:
            return "".join(chr(c) for c in value.flatten() if c != 0)
        except:
            pass

    return np.squeeze(value)

## Cell-level 메타데이터

Batch1의 46개 셀에서 `barcode`, `channel_id`, `policy`, `policy_readable`, `cycle_life`를 확인합니다. 분석용 표에는 `cell_id`, `channel`, `policy`, `cycle_life`만 남깁니다. `barcode`는 식별자라 모델 입력으로 바로 쓰지 않고, `policy`는 사람이 읽기 쉬운 `policy_readable`에서 가져옵니다. `cycle_life`는 이후 모델의 target입니다.

`channel_id`는 여섯 값으로 구성되며 Batch1에서는 다섯 번째 값인 `channel_4`만 셀마다 달라집니다. 따라서 현재 표의 `channel`로 사용하되 다른 배치에도 같다고 가정하지 않습니다.

In [85]:
# EDA에 필요한 셀 단위 필드만 골라 각 참조의 실제 값을 추출합니다.
fields = [
    "barcode",
    "channel_id",
    "policy",
    "policy_readable",
    "cycle_life",
]

data = {}

with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    for field in fields:
        values = []

        for ref in batch[field][:, 0]:
            values.append(read_h5_value(f, ref))

        data[field] = values

In [86]:
# 셀별 값으로 표를 만들고, Batch1 내부 순서인 cell_id와 정수형 수명을 추가합니다.
df_cells = pd.DataFrame(data)
df_cells["cycle_life"] = df_cells["cycle_life"].astype(int)

df_cells.insert(
    0,
    "cell_id",
    range(1, len(df_cells) + 1)
)

df_cells.head()

,cell_id,barcode,channel_id,policy,policy_readable,cycle_life
0,1,"[3707764736, 2, 1, 1, 1, 1]","[3707764736, 2, 1, 1, 47, 1]",3_6C-80PER_3_6C,3.6C(80%)-3.6C,1190
1,2,"[3707764736, 2, 1, 1, 2, 1]","[3707764736, 2, 1, 1, 48, 1]",3_6C-80PER_3_6C,3.6C(80%)-3.6C,1179
2,3,"[3707764736, 2, 1, 1, 3, 1]","[3707764736, 2, 1, 1, 49, 1]",3_6C-80PER_3_6C,3.6C(80%)-3.6C,1177
3,4,"[3707764736, 2, 1, 1, 4, 1]","[3707764736, 2, 1, 1, 50, 1]",4C-80PER_4C,4C(80%)-4C,1226
4,5,"[3707764736, 2, 1, 1, 5, 1]","[3707764736, 2, 1, 1, 51, 1]",4C-80PER_4C,4C(80%)-4C,1227


In [87]:
# channel_id의 여섯 요소를 분리해 Batch1에서 셀마다 달라지는 위치를 찾습니다.
channel_df = pd.DataFrame(
    df_cells["channel_id"].tolist(),
    columns=[
        "channel_0",
        "channel_1",
        "channel_2",
        "channel_3",
        "channel_4",
        "channel_5",
    ]
)

channel_df.head()

for col in channel_df.columns:
    print(
        col,
        "nunique =", channel_df[col].nunique(),
        "values =", channel_df[col].unique()[:10]
    )

channel_0 nunique = 1 values = [3707764736]
channel_1 nunique = 1 values = [2]
channel_2 nunique = 1 values = [1]
channel_3 nunique = 1 values = [1]
channel_4 nunique = 46 values = [47 48 49 50 51 52 53 54 55 56]
channel_5 nunique = 1 values = [1]


In [88]:
# 확인한 channel_4를 채널로 쓰고 EDA에 필요한 네 컬럼만 남깁니다.
df_cells["channel"] = channel_df["channel_4"]

df_eda = df_cells[
    [
        "cell_id",
        "channel",
        "policy_readable",
        "cycle_life"
    ]
].copy()

df_eda = df_eda.rename(
    columns={
        "policy_readable": "policy"
    }
)

df_eda["cycle_life"] = df_eda["cycle_life"].astype(int)

df_eda.head()

,cell_id,channel,policy,cycle_life
0,1,47,3.6C(80%)-3.6C,1190
1,2,48,3.6C(80%)-3.6C,1179
2,3,49,3.6C(80%)-3.6C,1177
3,4,50,4C(80%)-4C,1226
4,5,51,4C(80%)-4C,1227


In [89]:
# 다음 EDA 단계에서 재사용할 수 있도록 정리한 Batch1 셀 데이터를 CSV로 저장합니다.
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_eda.to_csv(
    PROCESSED_DIR / "batch1_cells.csv",
    index=False
)

## Cycle-level summary 구조

`summary`는 셀별 cycle 요약값입니다. 필드는 `IR`(내부저항), `QCharge`(충전 용량), `QDischarge`(방전 용량), `Tavg`(평균 온도), `Tmax`(최대 온도), `Tmin`(최소 온도), `chargetime`(충전 시간), `cycle`(cycle 번호)입니다. 첫 셀에는 1부터 1189까지의 cycle 요약 1189개가 있습니다. 첫 위치의 여러 측정값이 0이지만 이 단계에서 임의로 제거하지 않고 EDA에서 처리 여부를 판단합니다.

In [90]:
# 첫 셀의 summary 참조를 열어 cycle 단위 요약 필드 이름을 확인합니다.
with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    summary_ref = batch["summary"][0, 0]
    summary = f[summary_ref]

    print(type(summary))
    print(list(summary.keys()))

<class 'h5py._hl.group.Group'>
['IR', 'QCharge', 'QDischarge', 'Tavg', 'Tmax', 'Tmin', 'chargetime', 'cycle']


In [91]:
# 각 summary 필드의 길이와 자료형을 보고 행 단위 변환이 가능한지 확인합니다.
with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    summary_ref = batch["summary"][0, 0]
    summary = f[summary_ref]

    for key in summary.keys():
        obj = summary[key]

        print(
            f"{key:15} "
            f"type={type(obj).__name__:10} "
            f"shape={obj.shape} "
            f"dtype={obj.dtype}"
        )

IR              type=Dataset    shape=(1, 1189) dtype=float64
QCharge         type=Dataset    shape=(1, 1189) dtype=float64
QDischarge      type=Dataset    shape=(1, 1189) dtype=float64
Tavg            type=Dataset    shape=(1, 1189) dtype=float64
Tmax            type=Dataset    shape=(1, 1189) dtype=float64
Tmin            type=Dataset    shape=(1, 1189) dtype=float64
chargetime      type=Dataset    shape=(1, 1189) dtype=float64
cycle           type=Dataset    shape=(1, 1189) dtype=float64


In [92]:
# 첫 셀의 각 요약값을 확인하고 첫 위치의 0은 여기서 임의로 제거하지 않습니다.
with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    first_summary_ref = batch["summary"][0, 0]
    first_summary = f[first_summary_ref]

    for key in first_summary.keys():
        values = first_summary[key][()]
        values = values.squeeze()

        print(f"\n[{key}]")
        print("length:", len(values))
        print("first 10:", values[:10])
        print("last 10:", values[-10:])


[IR]
length: 1189
first 10: [0.         0.01674235 0.01672431 0.01668125 0.01666187 0.01662338
 0.01659958 0.0165769  0.0165882  0.01657218]
last 10: [0.0171504  0.01715273 0.01716297 0.01715394 0.01711801 0.01708356
 0.01707795 0.01708527 0.01707222 0.0170891 ]

[QCharge]
length: 1189
first 10: [0.        1.0710422 1.0716741 1.0723044 1.07297   1.0734913 1.073922
 1.0741584 1.0743366 1.0743531]
last 10: [1.026746  1.027101  1.0267582 1.0262543 1.0267483 1.0262955 1.0265589
 1.0263145 1.0266199 1.0260537]

[QDischarge]
length: 1189
first 10: [0.        1.0706892 1.0719005 1.0725095 1.073174  1.0735761 1.0739917
 1.0743741 1.0744922 1.0745373]
last 10: [1.0268544 1.0272599 1.0270069 1.0265244 1.0268919 1.026472  1.0267
 1.0264986 1.0266768 1.0262098]

[Tavg]
length: 1189
first 10: [ 0.         31.87501078 31.93149033 31.93260348 31.95932156 31.96106185
 31.9005621  31.92166785 31.87008232 31.84105268]
last 10: [31.99394015 31.87606157 31.96370184 31.97776213 32.19982599 32.30592278
 32

### Summary 길이와 target의 관계

46개 셀 모두에서 `summary_length = summary_last_cycle = cycle_life - 1`이 확인됩니다. 따라서 **summary 길이나 마지막 cycle 번호를 수명 예측 feature로 쓰면 target 정보가 새어 들어갑니다(data leakage).** 이후 특성 생성에서는 관측 시점을 먼저 정하고 그 시점까지의 측정값만 사용해야 합니다.

In [93]:
# 첫 셀에서 summary의 마지막 cycle과 최종 cycle_life가 어떻게 연결되는지 확인합니다.
with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    cycle_life_ref = batch["cycle_life"][0, 0]
    cycle_life = read_h5_value(f, cycle_life_ref)

    summary_ref = batch["summary"][0, 0]
    summary = f[summary_ref]

    cycles = summary["cycle"][()].squeeze()

    print("cycle_life:", cycle_life)
    print("summary length:", len(cycles))
    print("summary last cycle:", cycles[-1])

cycle_life: 1190.0
summary length: 1189
summary last cycle: 1189.0


In [94]:
# 같은 관계가 Batch1의 46개 셀 모두에서 성립하는지 검사합니다.
results = []

with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    n_cells = batch["cycle_life"].shape[0]

    for i in range(n_cells):
        cycle_life_ref = batch["cycle_life"][i, 0]
        cycle_life = read_h5_value(f, cycle_life_ref)

        summary_ref = batch["summary"][i, 0]
        summary = f[summary_ref]

        cycles = summary["cycle"][()].squeeze()

        results.append({
            "cell_id": i + 1,
            "cycle_life": int(cycle_life),
            "summary_length": len(cycles),
            "summary_last_cycle": int(cycles[-1]),
            "difference": int(cycle_life - cycles[-1])
        })

cycle_check_df = pd.DataFrame(results)

cycle_check_df.head()

,cell_id,cycle_life,summary_length,summary_last_cycle,difference
0,1,1190,1189,1189,1
1,2,1179,1178,1178,1
2,3,1177,1176,1176,1
3,4,1226,1225,1225,1
4,5,1227,1226,1226,1


In [95]:
# 차이값의 분포로 전체 셀에서 cycle_life - 마지막 cycle이 1인지 확인합니다.
cycle_check_df["difference"].value_counts()

difference
1    46
Name: count, dtype: int64

In [96]:
# 관계가 깨진 셀만 골라 예외가 있는지 확인합니다.
cycle_check_df[
    cycle_check_df["difference"] != 1
]

,cell_id,cycle_life,summary_length,summary_last_cycle,difference


## Summary를 긴 형식으로 저장

셀마다 다른 길이의 summary를 `cell_id`, `cycle`, `IR`, `QCharge`, `QDischarge`, `Tavg`, `Tmax`, `Tmin`, `chargetime` 열로 변환해 `data/processed/batch1_cycle_summary.csv`에 저장합니다. `policy`와 `cycle_life`는 `batch1_cells.csv`에 있으므로 중복 저장하지 않습니다.

두 CSV의 `cell_id`는 모두 1부터 시작해 그대로 병합할 수 있습니다.

In [97]:
# 셀마다 길이가 다른 summary를 cell_id와 cycle을 가진 긴 형식의 표로 모읍니다.
import h5py
import pandas as pd
import numpy as np

summary_rows = []

with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    n_cells = batch["summary"].shape[0]

    for cell_id in range(n_cells):
        # 각 cell의 summary reference 해제
        summary_ref = batch["summary"][cell_id, 0]
        summary = f[summary_ref]

        # summary 데이터 추출
        cycle = summary["cycle"][()].squeeze()
        IR = summary["IR"][()].squeeze()
        QCharge = summary["QCharge"][()].squeeze()
        QDischarge = summary["QDischarge"][()].squeeze()
        Tavg = summary["Tavg"][()].squeeze()
        Tmax = summary["Tmax"][()].squeeze()
        Tmin = summary["Tmin"][()].squeeze()
        chargetime = summary["chargetime"][()].squeeze()

        # 원본 인덱스는 0부터 시작하지만 저장 cell_id는 cells CSV처럼 1부터 시작합니다.
        # cycle별 행 생성
        cell_df = pd.DataFrame({
            "cell_id": cell_id + 1,
            "cycle": cycle,
            "IR": IR,
            "QCharge": QCharge,
            "QDischarge": QDischarge,
            "Tavg": Tavg,
            "Tmax": Tmax,
            "Tmin": Tmin,
            "chargetime": chargetime
        })

        summary_rows.append(cell_df)

# 46개 cell 하나로 합치기
batch1_cycle_summary = pd.concat(
    summary_rows,
    ignore_index=True
)

# cycle 번호는 정수형으로 변환
batch1_cycle_summary["cycle"] = (
    batch1_cycle_summary["cycle"].astype(int)
)

batch1_cycle_summary.head()

,cell_id,cycle,IR,QCharge,QDischarge,Tavg,Tmax,Tmin,chargetime
0,1,1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,1,2,0.016742,1.071042,1.070689,31.875011,35.652016,29.566130,13.341250
2,1,3,0.016724,1.071674,1.071900,31.931490,35.692978,29.604385,13.425777
3,1,4,0.016681,1.072304,1.072510,31.932603,35.680588,29.744202,13.425167
4,1,5,0.016662,1.072970,1.073174,31.959322,35.728691,29.644709,13.341442


In [98]:
# cycle-level EDA에 쓸 summary 표를 CSV로 저장합니다.
output_path = "../data/processed/batch1_cycle_summary.csv"

batch1_cycle_summary.to_csv(
    output_path,
    index=False
)

print(f"저장 완료: {output_path}")
print("shape:", batch1_cycle_summary.shape)

저장 완료: ../data/processed/batch1_cycle_summary.csv
shape: (38811, 9)


In [99]:
# 저장 전 행 수와 셀별 cycle 수를 확인해 long-form 변환 결과를 점검합니다.
print(batch1_cycle_summary.info())

print(
    batch1_cycle_summary
    .groupby("cell_id")
    .size()
    .head()
)

<class 'pandas.DataFrame'>
RangeIndex: 38811 entries, 0 to 38810
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   cell_id     38811 non-null  int64  
 1   cycle       38811 non-null  int64  
 2   IR          38811 non-null  float64
 3   QCharge     38811 non-null  float64
 4   QDischarge  38811 non-null  float64
 5   Tavg        38811 non-null  float64
 6   Tmax        38811 non-null  float64
 7   Tmin        38811 non-null  float64
 8   chargetime  38811 non-null  float64
dtypes: float64(7), int64(2)
memory usage: 2.7 MB
None
cell_id
1    1189
2    1178
3    1176
4    1225
5    1226
dtype: int64


## Cycle 내부 시계열 구조

`cycles`는 summary보다 세밀한 cycle별 시계열입니다. 첫 셀의 필드는 `I`, `Qc`, `Qd`, `Qdlin`, `T`, `Tdlin`, `V`, `discharge_dQdV`, `t`이며, 각 필드는 `(n_cycles, 1)` 형태의 HDF5 reference 배열입니다. 각 참조를 따라가야 해당 cycle의 실제 배열을 읽을 수 있습니다.

`I`, `Qc`, `Qd`, `T`, `V`, `t`는 실제 측정 시계열이라 cycle마다 길이가 다릅니다. `Qdlin`, `Tdlin`, `discharge_dQdV`는 정상 cycle에서 길이 1000인 보간·파생 profile로, 공통 전압축에서 비교할 수 있습니다.

In [100]:
# cycles 참조를 열어 summary보다 세밀한 cycle 내부 시계열 필드를 확인합니다.
with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    cycles_ds = batch["cycles"]

    print("cycles type:", type(cycles_ds).__name__)
    print("cycles shape:", cycles_ds.shape)
    print("cycles dtype:", cycles_ds.dtype)

    first_cycles_ref = cycles_ds[0, 0]
    first_cycles = f[first_cycles_ref]

    print("\n첫 번째 cell의 cycles 객체:")
    print(first_cycles)

    print("\ncycles 내부 keys:")
    print(list(first_cycles.keys()))

cycles type: Dataset
cycles shape: (46, 1)
cycles dtype: object

첫 번째 cell의 cycles 객체:
<HDF5 group "/#refs#/Td" (9 members)>

cycles 내부 keys:
['I', 'Qc', 'Qd', 'Qdlin', 'T', 'Tdlin', 'V', 'discharge_dQdV', 't']


In [101]:
# 각 시계열 필드가 cycle별 HDF5 reference 배열인지 크기와 dtype을 확인합니다.
with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    first_cycles_ref = batch["cycles"][0, 0]
    first_cycles = f[first_cycles_ref]

    for key in first_cycles.keys():
        obj = first_cycles[key]

        print(
            f"{key:20s}",
            f"type={type(obj).__name__:10s}",
            f"shape={obj.shape}",
            f"dtype={obj.dtype}"
        )

I                    type=Dataset    shape=(1189, 1) dtype=object
Qc                   type=Dataset    shape=(1189, 1) dtype=object
Qd                   type=Dataset    shape=(1189, 1) dtype=object
Qdlin                type=Dataset    shape=(1189, 1) dtype=object
T                    type=Dataset    shape=(1189, 1) dtype=object
Tdlin                type=Dataset    shape=(1189, 1) dtype=object
V                    type=Dataset    shape=(1189, 1) dtype=object
discharge_dQdV       type=Dataset    shape=(1189, 1) dtype=object
t                    type=Dataset    shape=(1189, 1) dtype=object


### Index 0의 dummy cycle

첫 셀의 모든 시계열 필드와 전체 46개 셀의 `I`에서 index 0은 길이 2의 `[0, 0]`입니다. 전체 셀의 모든 필드도 같은 값인지 별도 확인했으며, 원본의 공통 placeholder로 판단합니다. 실제 상세 시계열은 index 1 이후에 있습니다. 이후 EDA와 특성 생성에서는 index 0을 제외하되 원본 `.mat`은 수정하지 않습니다.

In [102]:
# 첫 셀의 index 0을 읽어 실제 측정치인지 placeholder인지 확인합니다.
with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    first_cycles_ref = batch["cycles"][0, 0]
    first_cycles = f[first_cycles_ref]

    for key in first_cycles.keys():
        ref = first_cycles[key][0, 0]
        values = f[ref][()].squeeze()

        print(f"\n[{key}]")
        print("shape:", values.shape)
        print("length:", len(values) if values.ndim > 0 else 1)
        print("first 10:", values[:10] if values.ndim > 0 else values)


[I]
shape: (2,)
length: 2
first 10: [0 0]

[Qc]
shape: (2,)
length: 2
first 10: [0 0]

[Qd]
shape: (2,)
length: 2
first 10: [0 0]

[Qdlin]
shape: (2,)
length: 2
first 10: [0 0]

[T]
shape: (2,)
length: 2
first 10: [0 0]

[Tdlin]
shape: (2,)
length: 2
first 10: [0 0]

[V]
shape: (2,)
length: 2
first 10: [0 0]

[discharge_dQdV]
shape: (2,)
length: 2
first 10: [0 0]

[t]
shape: (2,)
length: 2
first 10: [0 0]


In [103]:
# index 1의 시계열을 읽어 실제 cycle의 길이와 값 형태를 확인합니다.
with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    first_cycles_ref = batch["cycles"][0, 0]
    first_cycles = f[first_cycles_ref]

    cycle_idx = 1   # 두 번째 cycle

    for key in first_cycles.keys():
        ref = first_cycles[key][cycle_idx, 0]
        values = f[ref][()].squeeze()

        print(f"\n[{key}]")
        print("shape:", values.shape)
        print("length:", len(values) if values.ndim > 0 else 1)
        print("first 10:", values[:10] if values.ndim > 0 else values)


[I]
shape: (1087,)
length: 1087
first 10: [0.         0.21590753 0.35983085 0.46784575 0.5758771  0.6838391
 0.7553274  0.82793843 0.93612755 1.007726  ]

[Qc]
shape: (1087,)
length: 1087
first 10: [0.0000000e+00 1.0935055e-06 1.0935055e-06 1.0935055e-06 1.0935055e-06
 1.0935055e-06 1.0935055e-06 1.0935055e-06 1.0935055e-06 1.0935055e-06]

[Qd]
shape: (1087,)
length: 1087
first 10: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

[Qdlin]
shape: (1000,)
length: 1000
first 10: [-3.57431692e-04 -3.18524587e-04 -2.81852414e-04 -2.47206030e-04
 -2.14376291e-04 -1.83154053e-04 -1.53330172e-04 -1.24695504e-04
 -9.70409053e-05 -7.01572323e-05]

[T]
shape: (1087,)
length: 1087
first 10: [31.632496 31.632496 31.632496 31.632496 31.632496 31.632496 31.632496
 31.632496 31.632496 31.632496]

[Tdlin]
shape: (1000,)
length: 1000
first 10: [29.58588212 29.58588989 29.58589615 29.58590103 29.58590468 29.58590725
 29.58590886 29.58590966 29.5859098  29.58590942]

[V]
shape: (1087,)
length: 1087
first 10: [2.0264158 2

In [104]:
# 여러 cycle을 비교해 raw 길이는 달라지고 보간 profile 길이는 일정한지 확인합니다.
with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    first_cycles_ref = batch["cycles"][0, 0]
    first_cycles = f[first_cycles_ref]

    for cycle_idx in [0, 1, 2, 9, 99]:
        print(f"\ncycle index: {cycle_idx}")

        for key in first_cycles.keys():
            ref = first_cycles[key][cycle_idx, 0]
            values = f[ref][()].squeeze()

            length = len(values) if values.ndim > 0 else 1

            print(f"{key:20s} length={length}")


cycle index: 0
I                    length=2
Qc                   length=2
Qd                   length=2
Qdlin                length=2
T                    length=2
Tdlin                length=2
V                    length=2
discharge_dQdV       length=2
t                    length=2

cycle index: 1
I                    length=1087
Qc                   length=1087
Qd                   length=1087
Qdlin                length=1000
T                    length=1087
Tdlin                length=1000
V                    length=1087
discharge_dQdV       length=1000
t                    length=1087

cycle index: 2
I                    length=1117
Qc                   length=1117
Qd                   length=1117
Qdlin                length=1000
T                    length=1117
Tdlin                length=1000
V                    length=1117
discharge_dQdV       length=1000
t                    length=1117

cycle index: 9
I                    length=1130
Qc                   length=1130
Qd    

In [105]:
# 모든 셀에서 index 0의 I가 길이 2인 [0, 0]인지 점검합니다.
dummy_check = []

with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    for cell_id in range(batch["cycles"].shape[0]):
        cycles_ref = batch["cycles"][cell_id, 0]
        cycles = f[cycles_ref]

        ref = cycles["I"][0, 0]
        values = f[ref][()].squeeze()

        dummy_check.append({
            "cell_id": cell_id + 1,
            "cycle0_length": len(values),
            "cycle0_values": values.tolist()
        })

dummy_check_df = pd.DataFrame(dummy_check)

dummy_check_df.head()

,cell_id,cycle0_length,cycle0_values
0,1,2,"[0, 0]"
1,2,2,"[0, 0]"
2,3,2,"[0, 0]"
3,4,2,"[0, 0]"
4,5,2,"[0, 0]"


In [106]:
# index 0의 길이가 46개 셀에서 모두 같은지 집계합니다.
dummy_check_df["cycle0_length"].value_counts()

cycle0_length
2    46
Name: count, dtype: int64

## Vdlin 공통 전압축

`Vdlin`도 셀별 HDF5 reference로 저장됩니다. 첫 셀의 값은 3.5 V에서 2.0 V로 내려가는 1000개 점이며, 아래에서 46개 셀 모두 동일한지 확인합니다. `raw V/Qd/T`는 cycle마다 표본 수가 다르지만, `Vdlin`을 공통 축으로 삼은 `Qdlin`, `Tdlin`, `discharge_dQdV`는 같은 위치끼리 비교할 수 있는 profile로 판단합니다.

In [107]:
# Vdlin이 셀별 reference로 저장됐는지 먼저 확인합니다.
with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    vdlin = batch["Vdlin"]

    print("Vdlin type:", type(vdlin).__name__)
    print("Vdlin shape:", vdlin.shape)
    print("Vdlin dtype:", vdlin.dtype)

    print("\nVdlin raw object:")
    print(vdlin)

Vdlin type: Dataset
Vdlin shape: (46, 1)
Vdlin dtype: object

Vdlin raw object:
<HDF5 dataset "Vdlin": shape (46, 1), type "|O">


In [108]:
# Vdlin 첫 원소의 타입을 확인해 실제 전압축을 역참조해야 함을 확인합니다.
with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    vdlin = batch["Vdlin"]

    print("first element:", vdlin[0, 0])
    print("type of first element:", type(vdlin[0, 0]))

first element: <HDF5 object reference>
type of first element: <class 'h5py.h5r.Reference'>


In [109]:
# 첫 셀의 Vdlin 값과 길이를 보고 보간 profile의 전압축을 파악합니다.
with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    vdlin_ref = batch["Vdlin"][0, 0]
    vdlin_values = f[vdlin_ref][()].squeeze()

    print("shape:", vdlin_values.shape)
    print("length:", len(vdlin_values))
    print("first 10:", vdlin_values[:10])
    print("last 10:", vdlin_values[-10:])

shape: (1000,)
length: 1000
first 10: [3.5        3.4984985  3.496997   3.4954955  3.49399399 3.49249249
 3.49099099 3.48948949 3.48798799 3.48648649]
last 10: [2.01351351 2.01201201 2.01051051 2.00900901 2.00750751 2.00600601
 2.0045045  2.003003   2.0015015  2.        ]


## 데이터 수준별 필드 요약

| field | level | 역할 | 현재 판단 |
|---|---|---|---|
| `barcode` | cell | 셀 식별 정보 | 모델 feature 제외 후보 |
| `channel_id` | cell | 채널 식별 정보 | 실질 식별값은 `channel_4`, 모델 feature보다는 식별용 |
| `policy` / `policy_readable` | cell | 충전 정책 | 핵심 설명변수 후보 |
| `cycle_life` | cell | 최종 수명 | target |
| `cycle` | cycle-summary | cycle 번호 | 인덱스/축, feature 자체로는 사용하지 않음 |
| `IR` | cycle-summary | 내부저항 | 핵심 열화 feature 후보 |
| `QCharge` | cycle-summary | 충전 용량 | feature 후보 |
| `QDischarge` | cycle-summary | 방전 용량 | 핵심 열화 feature 후보 |
| `Tavg` | cycle-summary | 평균 온도 | feature 후보 |
| `Tmax` | cycle-summary | 최대 온도 | feature 후보 |
| `Tmin` | cycle-summary | 최소 온도 | feature 후보 |
| `chargetime` | cycle-summary | 충전 시간 | feature 후보 |
| `I` | cycle-timeseries | 전류 시계열 | raw measurement |
| `Qc` | cycle-timeseries | 누적 충전 용량 | raw measurement |
| `Qd` | cycle-timeseries | 누적 방전 용량 | raw measurement |
| `T` | cycle-timeseries | 온도 시계열 | raw measurement |
| `V` | cycle-timeseries | 전압 시계열 | raw measurement |
| `t` | cycle-timeseries | 시간축 | 시계열 축 |
| `Qdlin` | interpolated | `Vdlin` 기준 방전용량 | 고정 길이 profile, feature 후보 |
| `Tdlin` | interpolated | `Vdlin` 기준 온도 | 고정 길이 profile, feature 후보 |
| `discharge_dQdV` | derived | 방전 dQ/dV | 열화 profile feature 후보 |
| `Vdlin` | shared axis | 3.5V→2.0V, 1000-point 전압축 | feature가 아니라 기준축 |

## 구조 확인 결과

- Batch1에는 46개 cell이 존재한다.
- summary는 각 cell에 대해 cycle 1 ~ cycle_life-1의 cycle-level 요약값을 가진다.
- cycles의 index 0은 46개 cell 모두 [0, 0] 형태의 dummy/placeholder 데이터이다.
- 실제 상세 시계열은 index 1 이후부터 존재한다.
- raw timeseries(I, Qc, Qd, T, V, t)는 cycle마다 길이가 다르다.
- Qdlin, Tdlin, discharge_dQdV는 Vdlin 기준 1000-point 고정 길이 데이터이다.

In [110]:
# 46개 셀의 Vdlin이 같은 1000-point 전압축인지 비교합니다.
import h5py
import numpy as np
import pandas as pd

vdlin_list = []

with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    n_cells = batch["Vdlin"].shape[0]

    for cell_id in range(n_cells):
        vdlin_ref = batch["Vdlin"][cell_id, 0]
        vdlin_values = f[vdlin_ref][()].squeeze()

        vdlin_list.append(vdlin_values)

# 첫 번째 셀을 기준으로 모두 동일한지 확인
reference_vdlin = vdlin_list[0]

same_vdlin = []

for cell_id, vdlin in enumerate(vdlin_list):
    same = np.array_equal(reference_vdlin, vdlin)

    same_vdlin.append({
        "cell_id": cell_id + 1,
        "same_as_cell0": same,
        "length": len(vdlin)
    })

vdlin_check_df = pd.DataFrame(same_vdlin)

vdlin_check_df.head()

,cell_id,same_as_cell0,length
0,1,True,1000
1,2,True,1000
2,3,True,1000
3,4,True,1000
4,5,True,1000


In [111]:
# 공통 전압축과 일치하는 셀 수를 집계합니다.
vdlin_check_df["same_as_cell0"].value_counts()

same_as_cell0
True    46
Name: count, dtype: int64

In [112]:
# 다른 전압축을 가진 셀이 있는지 별도로 확인합니다.
vdlin_check_df[
    vdlin_check_df["same_as_cell0"] == False
]

,cell_id,same_as_cell0,length


In [113]:
# 모든 셀에 공통인 Vdlin을 별도 CSV로 저장해 profile의 축으로 사용합니다.
vdlin_df = pd.DataFrame({
    "Vdlin": reference_vdlin
})

vdlin_output_path = "../data/processed/batch1_vdlin.csv"

vdlin_df.to_csv(
    vdlin_output_path,
    index=False
)

print("저장 완료:", vdlin_output_path)
print("shape:", vdlin_df.shape)

저장 완료: ../data/processed/batch1_vdlin.csv
shape: (1000, 1)


## 고정 길이 profile 저장

각 셀의 `Qdlin`, `Tdlin`, `discharge_dQdV`에서 dummy index 0을 빼고 압축 NPZ에 저장합니다. NPZ key도 CSV와 같은 1부터 시작하는 `cell_id`를 씁니다. 예를 들어 `cell_1_cycle`의 cycle 번호와 `cell_1_qdlin`, `cell_1_tdlin`, `cell_1_dqdv`의 행이 순서대로 대응합니다. dummy cycle 1은 제외되어 cycle 번호 배열의 첫 값은 2입니다. 가변 길이 raw `I/V/T/t/Qc/Qd`는 전체 CSV로 저장하지 않고 필요할 때 `.mat`에서 읽습니다.

In [114]:
# dummy index 0을 제외하고 cycle별 고정 길이 profile을 셀 단위 배열로 모읍니다.
profiles = {}

with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    n_cells = batch["cycles"].shape[0]

    for cell_id in range(n_cells):
        cycles_ref = batch["cycles"][cell_id, 0]
        cycles = f[cycles_ref]
        summary_ref = batch["summary"][cell_id, 0]
        summary_cycles = f[summary_ref]["cycle"][()].squeeze().astype(int)[1:]

        qdlin_list = []
        tdlin_list = []
        dqdv_list = []

        n_cycles = cycles["Qdlin"].shape[0]
        if len(summary_cycles) != n_cycles - 1:
            raise ValueError(f"cell_id={cell_id + 1}: summary와 profile cycle 수가 다릅니다")

        # index 0은 dummy이므로 제외
        for cycle_idx in range(1, n_cycles):
            qdlin_ref = cycles["Qdlin"][cycle_idx, 0]
            tdlin_ref = cycles["Tdlin"][cycle_idx, 0]
            dqdv_ref = cycles["discharge_dQdV"][cycle_idx, 0]

            qdlin = f[qdlin_ref][()].squeeze()
            tdlin = f[tdlin_ref][()].squeeze()
            dqdv = f[dqdv_ref][()].squeeze()

            qdlin_list.append(qdlin)
            tdlin_list.append(tdlin)
            dqdv_list.append(dqdv)

        # profile의 각 행은 같은 위치의 cycle 번호와 짝을 이룹니다.
        profiles[f"cell_{cell_id + 1}_cycle"] = summary_cycles
        profiles[f"cell_{cell_id + 1}_qdlin"] = np.stack(qdlin_list)
        profiles[f"cell_{cell_id + 1}_tdlin"] = np.stack(tdlin_list)
        profiles[f"cell_{cell_id + 1}_dqdv"] = np.stack(dqdv_list)

In [115]:
# 큰 profile 배열은 CSV 대신 압축 NPZ로 저장합니다.
profiles_output_path = "../data/processed/batch1_cycle_profiles.npz"

np.savez_compressed(
    profiles_output_path,
    **profiles
)

print("저장 완료:", profiles_output_path)

저장 완료: ../data/processed/batch1_cycle_profiles.npz


In [116]:
# 저장된 NPZ를 다시 열어 셀별 세 profile의 key가 생성됐는지 확인합니다.
profiles = np.load(
    "../data/processed/batch1_cycle_profiles.npz"
)

print("저장된 key 개수:", len(profiles.files))
print("앞쪽 key:", profiles.files[:6])

저장된 key 개수: 184
앞쪽 key: ['cell_1_cycle', 'cell_1_qdlin', 'cell_1_tdlin', 'cell_1_dqdv', 'cell_2_cycle', 'cell_2_qdlin']


In [117]:
# 저장된 네 파일의 cell_id와 profile cycle 축이 맞는지 확인합니다.
saved_cells = pd.read_csv(PROCESSED_DIR / "batch1_cells.csv")
saved_summary = pd.read_csv(output_path)
saved_vdlin = pd.read_csv(vdlin_output_path)

assert saved_cells["cell_id"].tolist() == list(range(1, 47))
assert set(saved_summary["cell_id"]) == set(saved_cells["cell_id"])

with np.load(profiles_output_path) as saved_profiles:
    for cell in saved_cells.itertuples(index=False):
        cycle = saved_summary.loc[
            saved_summary["cell_id"] == cell.cell_id, "cycle"
        ].to_numpy()
        profile_cycle = saved_profiles[f"cell_{cell.cell_id}_cycle"]

        assert len(cycle) == cell.cycle_life - 1
        assert np.array_equal(profile_cycle, cycle[1:])
        for field in ("qdlin", "tdlin", "dqdv"):
            assert saved_profiles[f"cell_{cell.cell_id}_{field}"].shape == (
                len(profile_cycle), len(saved_vdlin)
            )

print("46개 셀의 ID·cycle·profile 축 확인 완료")

46개 셀의 ID·cycle·profile 축 확인 완료


## Processed 데이터와 다음 단계

```text
data/processed/
├── batch1_cells.csv             # cell-level metadata
├── batch1_cycle_summary.csv     # cycle-level summary
├── batch1_cycle_profiles.npz    # Qdlin / Tdlin / dQdV 고정 길이 profile
└── batch1_vdlin.csv             # profile의 공통 전압축
```

Batch1 구조와 cell-level, cycle-summary, cycle-timeseries, 보간 profile의 데이터 수준을 확인했습니다. 46개 셀에서 `summary_last_cycle = cycle_life - 1`이고, `cycles`의 index 0은 dummy입니다. `Vdlin`은 모든 셀에 공통인 1000-point 전압축이며 processed 파일이 생성되었습니다. 세 저장 데이터의 셀 ID는 모두 1부터 시작하고, NPZ의 `cell_N_cycle`은 profile 각 행의 cycle 번호를 담습니다. 다음 단계인 `02_eda.ipynb`에서는 cycle-level 열화 패턴을 탐색합니다. 수명 예측용 특성은 target 누출을 피하도록 초기 관측 구간을 정한 뒤 만들 예정입니다.